In [6]:
# import modules
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
#from scipy.stats import vonmises_fisher

# load the result object
from holopy.core.holopy_object import HoloPyObject, FullLoader
from holopy.core.utils import ensure_array, dict_without
import yaml
import importlib
# load using h5py
import h5py as h5

import holopy as hp
from holopy.core.process import normalize, bg_correct, center_find, subimage
from holopy.scattering import Sphere, Spheres, calc_holo
from holopy.inference import prior, ExactModel, CmaStrategy, EmceeStrategy, AlphaModel, NmpfitStrategy
from holopy.inference import model

In [7]:
#needed to make display work properly (there are other options as well if this fails)
%matplotlib tk

In [8]:
# path to directory
DIRECTORYPATH = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/'

# Load hologram from real data and fit it

In [15]:
# taken from Caroline's code
SPACING = 0.177
WAVELEN = 0.660
MEDIUM_INDEX = 1.33
POLARIZATION = POLARIZATION = [0.56, 0.83] # Calibrated 2020-09-02

#Sphere 1
R_1_MEAN = 0.6749016953839639
R_1_SIGMA =  0.00047233977835876834
N_1_MEAN =  1.5848484802283918
N_1_SIGMA =  0.00027320846407879903
#Sphere 2
R_2_MEAN =  0.6446649533295826
R_2_SIGMA =  0.000617682113114549
N_2_MEAN =  1.601784237444771
N_2_SIGMA =  0.0003353994780766957

DIMER_Z_GUESS = 20.0

In [16]:
# from Caroline's code used to load and process images
class Image():
    def __init__(self, rootdir, frame=None):
        '''
        Container to hold and process hologram images
        
        Parameters
        ----------
        rootdir: str or pathlib.Path
            directory containing .h5 file to load
        frame: int (optional)
            if included, extracts the given frame from the movie
        '''
        if isinstance(rootdir, str):
            rootdir = Path(rootdir)
        self.rootdir = rootdir
        self.bg = hp.core.io.load_average(str(rootdir / 'bg/'), spacing=SPACING)
        self.dc = hp.core.io.load_average(str(rootdir / 'dc/'), spacing=SPACING)
        self.raw_h5 = h5.File(rootdir / 'holo00.h5', 'r')
        if frame is not None:
            self.set_frame(frame)
    
    def set_frame(self, frame):
        '''
        Use a particular frame of the movie
        '''
        self.frame = frame
        try:
            image = self.raw_h5['images'][:, :, frame]
        except KeyError:
            # uncompressed h5
            image = np.array(self.raw_h5[str(frame)])
        self.raw = hp.core.metadata.data_grid(image, SPACING, MEDIUM_INDEX, WAVELEN, POLARIZATION)
        return self.raw
    
    def process(self):
        self.processed = normalize(bg_correct(self.raw, self.bg, self.dc))
        self.center = center_find(self.processed)
        return self.processed
    
    def crop(self, cropsize, center=None):
        '''
        Crops image to desired size
        
        Parameters
        ----------
        cropsize: int
            side length of resulting square image in pixels
        center: [flt, flt] (optional)
            if provided, it is used as the center for cropping. Otherwise center_find is used.
        '''
        self.process()
        if center is not None:
            self.center = center
        self.cropped = subimage(self.processed, self.center, cropsize)
        return self.cropped

In [17]:
def create_model(parameters):
    s1_r = parameters['r_1']
    s2_r = parameters['r_2']
    s1_n = parameters['n_1']
    s2_n = parameters['n_2']
    center_x = parameters['x_g']
    center_y = parameters['y_g']
    center_z = parameters['z_g']
    theta = parameters['theta']
    phi = parameters['phi']
    gap = parameters['gap']
    alpha = parameters['alpha']
    
    gap_center = np.array([center_x, center_y, center_z])
    components = np.array([np.cos(phi) * np.sin(theta), np.sin(phi) * np.sin(theta), np.cos(theta)])
    s1_center = gap_center + (s1_r + gap/2) * components
    s2_center = gap_center - (s2_r + gap/2) * components
        
    scatterer = Spheres([Sphere(r=s1_r, n=s1_n, center=s1_center),
                         Sphere(r=s2_r, n=s2_n, center=s2_center)], warn=False)
    return AlphaModel(scatterer, alpha=alpha)

In [29]:
# define model creation using KaiModel object
def create_kaimodel(parameters):
    s1_r = parameters['r_1']
    s2_r = parameters['r_2']
    s1_n = parameters['n_1']
    s2_n = parameters['n_2']
    center_x = parameters['x_g']
    center_y = parameters['y_g']
    center_z = parameters['z_g']
    theta = parameters['theta']
    phi = parameters['phi']
    gap = parameters['gap']
    alpha = parameters['alpha']
    
    gap_center = np.array([center_x, center_y, center_z])
    components = np.array([np.cos(phi) * np.sin(theta), np.sin(phi) * np.sin(theta), np.cos(theta)])
    s1_center = gap_center + (s1_r + gap/2) * components
    s2_center = gap_center - (s2_r + gap/2) * components
        
    scatterer = Spheres([Sphere(r=s1_r, n=s1_n, center=s1_center),
                         Sphere(r=s2_r, n=s2_n, center=s2_center)], warn=False)
    return model.KaiModel(scatterer, alpha=alpha)

In [27]:
# load path for data
path = '/Volumes/manoharan_lab/cmartin/Data/09-21-21/depletion/0.0875/03/'

In [33]:
# path for saving data
SAVEPATH = DIRECTORYPATH + 'real_data_0.0875/03/'

# Load Caroline's result4 (lm fit) in order to use this in priors

In [20]:
# Load one of Caroline's fits
Caroline_load_path = '/Volumes/manoharan_lab/cmartin/Fits/depletion_holography/09-21-21/depletion/0.0875/03/mcmcdimer_frame0_lm.h5'
Caroline_fit = hp.load(Caroline_load_path)

In [21]:
# set results4 to the loaded fit
results4 = Caroline_fit

In [23]:
print(results4.parameters['theta'])
print(results4.parameters['phi'])
print(results4.parameters['x_g'])

2.6507477109128157
6.332088090077169
58.506394342058385


# Fit data hologram using my model

In [30]:
# just do frame 0 for now
frames = [0]
i = 0
img = Image(path, frame=frames[i])
dimer_holo = img.crop(200)
x, y = img.center * SPACING
    
# Step 5: Define MCMC priors and run MCMC
r_1 = prior.BoundedGaussian(R_1_MEAN, R_1_SIGMA, lower_bound=0, upper_bound=1.0, name="r_1")
r_2 = prior.BoundedGaussian(R_2_MEAN, R_2_SIGMA, lower_bound=0, upper_bound=1.0, name="r_2")
n_1 = prior.BoundedGaussian(N_1_MEAN, N_1_SIGMA, lower_bound=0, upper_bound=1.7, name="n_1")
n_2 = prior.BoundedGaussian(N_2_MEAN, N_2_SIGMA, lower_bound=0, upper_bound=1.7, name="n_2")
x_g = prior.BoundedGaussian(results4.parameters['x_g'], SPACING, lower_bound=(x-5), 
                            upper_bound=(x+5), name="x_g")
y_g = prior.BoundedGaussian(results4.parameters['y_g'], SPACING, lower_bound=(y-5), 
                            upper_bound=(y+5), name="y_g")
z_g = prior.BoundedGaussian(results4.parameters['z_g'], 1, lower_bound=0, upper_bound=50, name="z_g")

# Caroline's code for normal model
#theta = prior.BoundedGaussian(results4.parameters['theta'], 0.1, lower_bound=0-0.1, 
                              #upper_bound=np.pi+0.1, name="theta")
#phi = prior.BoundedGaussian(results4.parameters['phi'], 0.1, lower_bound=0-0.1, 
                            #upper_bound=2*np.pi+0.1, name="phi")

# New priors for using kaimodel
# note: k argument needs to be the same for both theta and phi
theta = prior.Theta(5, results4.parameters['theta'], name="theta")
phi = prior.Phi(5, results4.parameters['phi'], name="phi")

gap = prior.BoundedGaussian(results4.parameters['gap'], 0.005, lower_bound=0, 
                            upper_bound=R_1_MEAN, name="gap")
alpha = prior.BoundedGaussian(results4.parameters['alpha'], 0.5, lower_bound=0.5, 
                              upper_bound=1.2, name="alpha") 
step_5_parameters = {'r_1': r_1, 'r_2': r_2, 'n_1': n_1, 'n_2': n_2,
                     'x_g': x_g, 'y_g': y_g, 'z_g': z_g,
                     'theta': theta, 'phi': phi, 'gap': gap, 'alpha': alpha}
model5 = create_kaimodel(step_5_parameters)

In [31]:
# now set initial conditions

# originally 50 walkers but start with 30 for speed (turns out this might be degrading performance)
nwalkers = 50
nsamples = 1000 # default is 1000

# originally used model5.generate_guess(nwalkers, scaling=0.1) but need different method
# before we implement .generate_guess for joint von Mises_Fisher
initial_guess = np.zeros((nwalkers, len(model5._parameters)))
for n in range(nwalkers):
    means = []
    # add some variance in starting point based on variance in distributions
    # need to make sure to avoid unphysical starting positions
    # added more variance to some and less to others (angles) to hopefully get faster convergence
    # originally it was p.sd*0.5
    scaling = 0.1
    for p in model5._parameters:
        if p.name == 'theta':
            means.append(p.mu + scaling*(np.random.normal(p.mu,0.1)-p.mu))
        elif p.name == 'phi':
            means.append(p.mu + scaling*(np.random.normal(p.mu,0.1)-p.mu))
        #elif p.name == 'x_g':
            #means.append(p.mu + scaling*np.random.normal(0,0.177))
        #elif p.name == 'y_g':
            #means.append(p.mu + scaling*np.random.normal(0,0.177))
        #elif p.name == 'z_g':
            #means.append(p.mu + scaling*np.random.normal(0,1))
        else:
            means.append(p.mu + scaling*(np.random.normal(p.mu,p.sd)-p.mu))
            
    initial_guess[n,:] = means
    
emcee_strategy = EmceeStrategy(npixels=8000, nwalkers=nwalkers,nsamples=nsamples, walker_initial_pos=initial_guess)

# Run fit

In [32]:
results5 = hp.sample(dimer_holo, model5, strategy=emcee_strategy)
hp.save(SAVEPATH+'dimer_frame'+str(frames[i])+'_mcmc.h5', results5)

print('von Mises-Fisher angles fit completed')
print(results5.guess_parameters)
print(results5.parameters)

NameError: name 'SAVEPATH' is not defined

# Run fit with initial conditions that match means of previous fit

In [83]:
# make new model with priors centered on results of last fit
frames = [0]
i = 0
img = Image(path, frame=frames[i])
dimer_holo = img.crop(200)
x, y = img.center * SPACING
    
# Step 6: Define MCMC priors and run MCMC using results of previous MCMC
r_1 = prior.BoundedGaussian(results5.parameters['r_1'], R_1_SIGMA, lower_bound=0, upper_bound=1.0, name="r_1")
r_2 = prior.BoundedGaussian(results5.parameters['r_2'], R_2_SIGMA, lower_bound=0, upper_bound=1.0, name="r_2")
n_1 = prior.BoundedGaussian(results5.parameters['n_1'], N_1_SIGMA, lower_bound=0, upper_bound=1.7, name="n_1")
n_2 = prior.BoundedGaussian(results5.parameters['n_2'], N_2_SIGMA, lower_bound=0, upper_bound=1.7, name="n_2")
x_g = prior.BoundedGaussian(results5.parameters['x_g'], SPACING, lower_bound=(x-5), 
                            upper_bound=(x+5), name="x_g")
y_g = prior.BoundedGaussian(results5.parameters['y_g'], SPACING, lower_bound=(y-5), 
                            upper_bound=(y+5), name="y_g")
z_g = prior.BoundedGaussian(results5.parameters['z_g'], 1, lower_bound=0, upper_bound=50, name="z_g")

# Caroline's code for normal model
#theta = prior.BoundedGaussian(results4.parameters['theta'], 0.1, lower_bound=0-0.1, 
                              #upper_bound=np.pi+0.1, name="theta")
#phi = prior.BoundedGaussian(results4.parameters['phi'], 0.1, lower_bound=0-0.1, 
                            #upper_bound=2*np.pi+0.1, name="phi")

# New priors for using kaimodel
# note: k argument needs to be the same for both theta and phi
theta = prior.Theta(5, results5.parameters['theta'], name="theta")
phi = prior.Phi(5, results5.parameters['phi'], name="phi")

gap = prior.BoundedGaussian(results5.parameters['gap'], 0.005, lower_bound=0, 
                            upper_bound=R_1_MEAN, name="gap")
alpha = prior.BoundedGaussian(results5.parameters['alpha'], 0.5, lower_bound=0.5, 
                              upper_bound=1.2, name="alpha") 
step_6_parameters = {'r_1': r_1, 'r_2': r_2, 'n_1': n_1, 'n_2': n_2,
                     'x_g': x_g, 'y_g': y_g, 'z_g': z_g,
                     'theta': theta, 'phi': phi, 'gap': gap, 'alpha': alpha}
model6 = create_kaimodel(step_6_parameters)

In [87]:
# now set initial conditions

# originally 50 walkers but start with 30 for speed (turns out this might be degrading performance)
nwalkers = 50
nsamples = 1000 # default is 1000

# originally used model5.generate_guess(nwalkers, scaling=0.1) but need different method
# before we implement .generate_guess for joint von Mises_Fisher
initial_guess = np.zeros((nwalkers, len(model6._parameters)))
for n in range(nwalkers):
    means = []
    # add some variance in starting point based on variance in distributions
    # need to make sure to avoid unphysical starting positions
    # added more variance to some and less to others (angles) to hopefully get faster convergence
    # originally it was p.sd*0.5
    scaling = 0.1
    for p in model6._parameters:
        if p.name == 'theta':
            means.append(p.mu + scaling*(np.random.normal(p.mu,0.1)-p.mu))
        elif p.name == 'phi':
            means.append(p.mu + scaling*(np.random.normal(p.mu,0.1)-p.mu))
        #elif p.name == 'x_g':
            #means.append(p.mu + scaling*np.random.normal(0,0.177))
        #elif p.name == 'y_g':
            #means.append(p.mu + scaling*np.random.normal(0,0.177))
        #elif p.name == 'z_g':
            #means.append(p.mu + scaling*np.random.normal(0,1))
        else:
            means.append(p.mu + scaling*(np.random.normal(p.mu,p.sd)-p.mu))
            
    initial_guess[n,:] = means
    
emcee_strategy6 = EmceeStrategy(npixels=8000, nwalkers=nwalkers,nsamples=nsamples, walker_initial_pos=initial_guess)

In [88]:
print(model6)
print(emcee_strategy6)

KaiModel(_dummy_scatterer=Spheres(scatterers=[Sphere(n=0, r=0, center=[0, 0, 0]), Sphere(n=0, r=0, center=[0, 0, 0])], warn=False), theory=Multisphere(niter=200, eps=1e-06, meth=1, qeps1=1e-05, qeps2=1e-08, compute_escat_radial=False, suppress_fortran_output=True), _parameters=[BoundedGaussian(mu=1.5837615975996668, sd=0.00027320846407879903, lower_bound=0, upper_bound=1.7, name='n_1'), BoundedGaussian(mu=0.666459947416345, sd=0.00047233977835876834, lower_bound=0, upper_bound=1.0, name='r_1'), BoundedGaussian(mu=58.484832600004395, sd=0.177, lower_bound=53.41128113511029, upper_bound=63.41128113511029, name='x_g'), BoundedGaussian(mu=0.11289545130974342, sd=0.005, lower_bound=0, upper_bound=0.6749016953839639, name='gap'), Phi(mu=6.3318712339240255, name='phi', sd=1), Theta(mu=2.6508866488913227, name='theta', sd=1), BoundedGaussian(mu=108.65291035266634, sd=0.177, lower_bound=103.65935236672793, upper_bound=113.65935236672793, name='y_g'), BoundedGaussian(mu=21.054742080951772, sd=1,

In [95]:
results6 = hp.sample(dimer_holo, model6, strategy=emcee_strategy6)
hp.save(SAVEPATH+'dimer_frame'+str(frames[i])+'_mcmc.h5', results6)

print('von Mises-Fisher angles fit completed')
print(results6.guess_parameters)
print(results6.parameters)

KeyboardInterrupt: 

In [34]:
#hp.save(SAVEPATH+'dimer_frame'+str(frames[i])+'_mcmc.h5', results5)

/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/xarray/core/common.py:615: FutureWarning: Updating MultiIndexed coordinate 'point' would corrupt indices for other variables: ['x', 'y', 'z']. This will raise an error in the future. Use `.drop_vars({'z', 'y', 'point', 'x'})` before assigning new coordinate values.
  data.coords.update(results)


In [94]:
SAVEPATH = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/real_data_0.0875/03' + '/fit_with_priors_centered_on_previous_mcmc_fit_results/'
print(SAVEPATH)
hp.save(SAVEPATH+'dimer_frame'+str(frames[i])+'_mcmc.h5', results6)


/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/real_data_0.0875/03/fit_with_priors_centered_on_previous_mcmc_fit_results/


## Load fit if necessary

In [7]:
# path that determines what fit you load
results_path = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/normal_side_by_side_geometry/variable_initial_conditions_version_1/longer_fits/von_Mises_Fisher_nsample_2000_fit_1_mcmc.h5'

In [10]:
INITIALCONDPATH = GEOMETRYPATH + 'no_mod_initial_conditions_version_1/particle_swap_not_degenerate/'

In [11]:
# alternative way of getting results path (less explicit)
LOAD_NAME_OF_FIT = 'von_Mises_Fisher_walkers_50_nsample_1000_fit_1'
results_path = INITIALCONDPATH + LOAD_NAME_OF_FIT + '_mcmc.h5'

In [62]:
# Load one of Caroline's fits
Caroline_load_path = '/Volumes/manoharan_lab/cmartin/Fits/depletion_holography/09-21-21/depletion/0.0875/mcmcdimer_frame0_mcmc.h5'
Caroline_fit = hp.load(Caroline_load_path)

/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/compare_Caroline_fit/depletion_holography_09-21-21_depletion_0.0875_mcmcdimer_frame0


In [63]:
SAVEPATH = DIRECTORYPATH+ 'compare_Caroline_fit/depletion_holography_09-21-21_depletion_0.0875_mcmcdimer_frame0'
print(SAVEPATH)
results5 = Caroline_fit

/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/compare_Caroline_fit/depletion_holography_09-21-21_depletion_0.0875_mcmcdimer_frame0


In [16]:
# try to load fit result object using hp.load code
# may need to modify code now that noise_sd is assigned as an attribute instead of a coord
attr_coords = '_attr_coords'
def unpack_attrs(a):
    if len(a) == 0:
        return a
    new_attrs={}
    attr_ref = yaml.load(a[attr_coords], Loader=FullLoader)
    attrs_to_ignore = ['spacing', 'name', '_dummy_channel', '_image_scaling']
    for attr in dict_without(attr_ref, attrs_to_ignore):
        if attr_ref[attr]:
            new_attrs[attr] = xr.DataArray(
                a[attr],
                coords=attr_ref[attr],
                dims=list(attr_ref[attr].keys()))
        elif attr in a:
            new_attrs[attr] = yaml.safe_load(a[attr])
        else:
            new_attrs[attr] = None
    return new_attrs

with xr.open_dataset(results_path, engine='h5netcdf') as ds:
    if '_source_class' in ds.attrs:
        _source_class = ds.attrs.pop('_source_class')
        pathtok = _source_class.split('.')
        cls = getattr(importlib.import_module(".".join(pathtok[:-1])), pathtok[-1])
        #ds.close()
        #return_variable = cls._load(results_path)
    #def _load(cls, ds, **kwargs):
        #with xr.open_dataset(ds, engine='h5netcdf', **kwargs) as ds:
        print(ds.load())
        dataset = ds
        data = dataset.data
        data.attrs = unpack_attrs(data.attrs)
        # Kai added this to move noise_sd to correct place
        data.attrs['noise_sd'] = data.coords['noise_sd'].to_numpy()
        if '_flat' in data.attrs.keys():
            flats = np.array(data.attrs['_flat']).T
            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
            codes = [[level.index(f) for f in flat]
                     for level, flat in zip(levels, flats)]
            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
            coordnames = list(data.coords)
            coordnames.remove('point')
            # seems like noise_sd should be attribute not coordinate
            coordnames.remove('noise_sd') # added this
            coords = {coord: data[coord] for coord in coordnames}
            coords['flat'] = flat_index
            data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                coords=coords, attrs=data.attrs)
            print(data)
        print(dataset.attrs['model'])
        # seems like model is a problem because I have an sd attribute in the priors?
        #model = yaml.load(dataset.attrs['model'], Loader=FullLoader)
        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
        outlist = [data, model, strategy]
        outlist.append(yaml.safe_load(dataset.attrs['time']))
        kwargs = yaml.safe_load(dataset.attrs['_kwargs'])
        for key in ['lnprobs', 'samples', '_best_fit']:
            try:
                kwargs[key] = getattr(dataset, key)
                kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
            except AttributeError:
                pass
        outlist.append(kwargs)
        # return args
        args = outlist
        
        #args = cls._unserialize(ds.load())
        return_variable = cls(*args)
'''
def _unserialize(cls, dataset):
        data = dataset.data
        data.attrs = unpack_attrs(data.attrs)
        if '_flat' in data.attrs.keys():
            flats = np.array(data.attrs['_flat']).T
            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
            codes = [[level.index(f) for f in flat]
                     for level, flat in zip(levels, flats)]
            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
            coordnames = list(data.coords)
            coordnames.remove('point')
            coords = {coord: data[coord] for coord in coordnames}
            coords['flat'] = flat_index
            data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                coords=coords, attrs=data.attrs)
        model = yaml.load(dataset.attrs['model'], Loader=FullLoader)
        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
        outlist = [data, model, strategy]
        outlist.append(yaml.safe_load(dataset.attrs['time']))
        kwargs = yaml.safe_load(dataset.attrs['_kwargs'])

        for key in ['lnprobs', 'samples', '_best_fit']:
            try:
                kwargs[key] = getattr(dataset, key)
                kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
            except AttributeError:
                pass
        outlist.append(kwargs)
        return outlist
'''

<xarray.Dataset>
Dimensions:    (point: 8000, walker: 50, chain: 1000, parameter: 11)
Coordinates:
  * point      (point) int64 0 1 2 3 4 5 6 ... 7994 7995 7996 7997 7998 7999
  * parameter  (parameter) object 'n_1' 'r_1' 'x_g' ... 'n_2' 'r_2' 'alpha'
Dimensions without coordinates: walker, chain
Data variables:
    data       (point) float64 0.909 1.208 1.037 1.025 ... 1.007 0.933 1.019
    lnprobs    (walker, chain) float64 -7.421e+04 -7.421e+04 ... 2.67e+04
    samples    (walker, chain, parameter) float64 1.585 0.675 ... 0.6448 0.9994
Attributes:
    model:     !KaiModel\n_dummy_scatterer: !Spheres\n  scatterers: [!Sphere ...
    strategy:  !EmceeStrategy\nnwalkers: 50\nnsamples: 1000\nnpixels: 8000\nw...
    time:      2127.895318031311
    _kwargs:   {}\n


KeyError: 'noise_sd'

In [12]:
# modified since noise_sd is assigned as an attribute instead of a coord
attr_coords = '_attr_coords'
def unpack_attrs(a):
    if len(a) == 0:
        return a
    new_attrs={}
    attr_ref = yaml.load(a[attr_coords], Loader=FullLoader)
    attrs_to_ignore = ['spacing', 'name', '_dummy_channel', '_image_scaling']
    for attr in dict_without(attr_ref, attrs_to_ignore):
        if attr_ref[attr]:
            new_attrs[attr] = xr.DataArray(
                a[attr],
                coords=attr_ref[attr],
                dims=list(attr_ref[attr].keys()))
        elif attr in a:
            new_attrs[attr] = yaml.safe_load(a[attr])
        else:
            new_attrs[attr] = None
    return new_attrs

with xr.open_dataset(results_path, engine='h5netcdf') as ds:
    if '_source_class' in ds.attrs:
        _source_class = ds.attrs.pop('_source_class')
        pathtok = _source_class.split('.')
        cls = getattr(importlib.import_module(".".join(pathtok[:-1])), pathtok[-1])
        #ds.close()
        #return_variable = cls._load(results_path)
    #def _load(cls, ds, **kwargs):
        #with xr.open_dataset(ds, engine='h5netcdf', **kwargs) as ds:
        print(ds.load())
        dataset = ds
        data = dataset.data
        data.attrs = unpack_attrs(data.attrs)
        if '_flat' in data.attrs.keys():
            flats = np.array(data.attrs['_flat']).T
            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
            codes = [[level.index(f) for f in flat]
                     for level, flat in zip(levels, flats)]
            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
            coordnames = list(data.coords)
            coordnames.remove('point')
            coords = {coord: data[coord] for coord in coordnames}
            coords['flat'] = flat_index
            data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                coords=coords, attrs=data.attrs)
            print(data)
        print(dataset.attrs['model'])
        # seems like model is a problem because I have an sd attribute in the priors?
        #model = yaml.load(dataset.attrs['model'], Loader=FullLoader)
        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
        outlist = [data, model, strategy]
        outlist.append(yaml.safe_load(dataset.attrs['time']))
        kwargs = yaml.safe_load(dataset.attrs['_kwargs'])
        for key in ['lnprobs', 'samples', '_best_fit']:
            try:
                kwargs[key] = getattr(dataset, key)
                kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
            except AttributeError:
                pass
        outlist.append(kwargs)
        # return args
        args = outlist
        
        #args = cls._unserialize(ds.load())
        return_variable = cls(*args)

<xarray.Dataset>
Dimensions:    (point: 8000, walker: 50, chain: 1000, parameter: 11)
Coordinates:
  * point      (point) int64 0 1 2 3 4 5 6 ... 7994 7995 7996 7997 7998 7999
  * parameter  (parameter) object 'n_1' 'r_1' 'x_g' ... 'n_2' 'r_2' 'alpha'
Dimensions without coordinates: walker, chain
Data variables:
    data       (point) float64 0.9506 0.9849 1.008 1.027 ... 0.917 1.023 1.041
    lnprobs    (walker, chain) float64 1.431e+04 1.431e+04 ... 2.671e+04
    samples    (walker, chain, parameter) float64 1.585 0.6749 ... 0.645 0.9992
Attributes:
    model:     !KaiModel\n_dummy_scatterer: !Spheres\n  scatterers: [!Sphere ...
    strategy:  !EmceeStrategy\nnwalkers: 50\nnsamples: 1000\nnpixels: 8000\nw...
    time:      2280.9646060466766
    _kwargs:   {}\n
<xarray.DataArray (flat: 8000)>
array([0.95056503, 0.98485299, 1.00834467, ..., 0.91699669, 1.0232044 ,
       1.04089093])
Coordinates:
  * flat     (flat) object MultiIndex
  * x        (flat) float64 11.15 6.372 15.93 2.124

In [14]:
# seems like don't need model at least for current analysis
# end up with noise_sd as a coordinate which is weird
print(return_variable)
samples = return_variable.samples[:,999]
lnprob = return_variable.lnprobs[5]
# can get rid of noise_sd coord using .reset_coords('noise_sd', drop = True)
print(samples.reset_coords('noise_sd',drop=True))
print(lnprob.reset_coords('noise_sd',drop=True))
burnt_samples = return_variable.burn_in(110).samples[:,889]

SamplingResult(data=<xarray.DataArray (flat: 8000)>
array([0.95056503, 0.98485299, 1.00834467, ..., 0.91699669, 1.0232044 ,
       1.04089093])
Coordinates:
  * flat     (flat) object MultiIndex
  * x        (flat) float64 11.15 6.372 15.93 2.124 ... 16.64 0.885 12.39 11.86
  * y        (flat) float64 1.593 16.64 12.39 11.68 ... 15.4 5.31 5.31 2.301
  * z        (flat) int64 0 0 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0 0
Attributes:
    _flat:               [[11.151, 1.593, 0], [6.372, 16.637999999999998, 0],...
    illum_polarization:  <xarray.DataArray (vector: 3)>\narray([0.55930131, 0...
    illum_wavelen:       0.66
    medium_index:        1.33
    noise_sd:            0.00862558
    original_dims:       {'x': [0.0, 0.177, 0.354, 0.5309999999999999, 0.708,..., model=<module 'holopy.inference.model' from '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holopy_code/holopy/holopy/inference/model.py'>, strategy=EmceeStrategy(nwalkers=50, nsamples=1000, npixels=8000, w

ValueError: One or more of the specified variables cannot be found in this dataset

In [15]:
# set results equal to loaded fit for further analysis
results5 = return_variable

## Work on futher data processing (ie drop pre-burn in data, drop non-converged fits, and decimate remaining data so its independent)

In [16]:
# set save path for figures (only necessary if loaded fit and not from earlier)
SAVEPATH = INITIALCONDPATH + LOAD_NAME_OF_FIT
print(SAVEPATH)

/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/on_top_of_each_other_geometry_with_noise/no_mod_initial_conditions_version_1/particle_swap_not_degenerate/von_Mises_Fisher_walkers_50_nsample_1000_fit_1


In [96]:
samples = results6.samples
print(samples[:,999][0])
print(means)

<xarray.DataArray (parameter: 11)>
array([1.58338314e+00, 6.61810399e-01, 5.84937320e+01, 1.08363985e-01,
       6.33215265e+00, 2.65374732e+00, 1.08652212e+02, 2.10359701e+01,
       1.59971242e+00, 6.53749957e-01, 6.43446763e-01])
Coordinates:
  * parameter  (parameter) <U5 'n_1' 'r_1' 'x_g' 'gap' ... 'n_2' 'r_2' 'alpha'
Attributes:
    acceptance_fraction:  0.4101799999999999
[1.583726196514745, 0.6664235732985598, 58.46192307204503, 0.11316391015815017, 6.3410849043146085, 2.660298049709442, 108.6570280816238, 21.026538216193, 1.6002167068691273, 0.649077996672116, 0.688996312318098]


In [97]:
print(samples[:,0].sel(parameter='phi'))

<xarray.DataArray (walker: 50)>
array([6.33720825, 6.33998815, 6.32349877, 6.34625714, 6.33353435,
       6.33913019, 6.33465505, 6.33372   , 6.33424197, 6.34160309,
       6.34008844, 6.32893577, 6.31841436, 6.33022361, 6.33423525,
       6.34057469, 6.33651475, 6.32733547, 6.33930499, 6.33368493,
       6.3382333 , 6.34473374, 6.34288955, 6.30700706, 6.33600083,
       6.33009705, 6.33390416, 6.33094803, 6.31835266, 6.337107  ,
       6.33199072, 6.32606402, 6.33986456, 6.32442276, 6.33453785,
       6.32500346, 6.32547301, 6.33129404, 6.33308461, 6.3386144 ,
       6.3231174 , 6.32333075, 6.33226935, 6.35098389, 6.31768577,
       6.32398468, 6.32664377, 6.33851072, 6.34253842, 6.34060844])
Coordinates:
    parameter  <U5 'phi'
Dimensions without coordinates: walker
Attributes:
    acceptance_fraction:  0.4101799999999999


In [41]:
print(initial_guess[0])

[1.58486443e+00 6.74989916e-01 5.85135853e+01 9.89276102e-02
 6.33477163e+00 2.65102329e+00 1.08650458e+02 2.10216152e+01
 1.60176563e+00 6.44553454e-01 6.20269077e-01]


In [98]:
# select gap parameter values for first walker (all chains)
gaps_example = samples[1].sel(parameter='gap')
print(gaps_example)

<xarray.DataArray (chain: 1000)>
array([0.11339539, 0.11317322, 0.11317322, 0.11317322, 0.11317322,
       0.11303293, 0.11298533, 0.11298421, 0.11306678, 0.11305209,
       0.11305209, 0.11303588, 0.1130036 , 0.1130036 , 0.1130036 ,
       0.1130036 , 0.11300245, 0.11300409, 0.11300399, 0.11300399,
       0.11300385, 0.11300385, 0.11300385, 0.11300385, 0.11308863,
       0.11307389, 0.11307389, 0.11306946, 0.11306946, 0.11306158,
       0.11306158, 0.11304786, 0.11304786, 0.11304786, 0.11297055,
       0.11294905, 0.11296492, 0.11296492, 0.11296492, 0.11296492,
       0.11293043, 0.11294824, 0.11294824, 0.11294824, 0.1128522 ,
       0.1128522 , 0.1128522 , 0.11282857, 0.11282857, 0.11282857,
       0.11282857, 0.11282857, 0.1127962 , 0.1127962 , 0.11278876,
       0.11263655, 0.11263655, 0.11263655, 0.11263655, 0.11263655,
       0.11263595, 0.11260948, 0.11269385, 0.11269385, 0.11264795,
       0.11264795, 0.11264795, 0.11264795, 0.11258174, 0.11258174,
       0.11254328, 0.11254328

In [99]:
plt.figure()
plt.plot(gaps_example)

### Drop pre-burn in (right now ad hoc but come back to make more systematic)

In [100]:
# plot pre-burn in
plt.figure()
plt.title('Pre burn-in logprob of fit')
plt.ylabel('lnprob')
plt.xlabel('sample (chain)')
for i in range(11):
    plt.plot(results6.lnprobs[i])
plt.savefig(SAVEPATH + '/pre_burn_in_lnprob.png')

In [233]:
for i in range(4):
    #plt.plot(results5.lnprobs[i])
    print(results5.lnprobs[:,999][i])

<xarray.DataArray ()>
array(30694.85396699)
Attributes:
    acceptance_fraction:  0.3945333333333333
<xarray.DataArray ()>
array(30697.28410234)
Attributes:
    acceptance_fraction:  0.3945333333333333
<xarray.DataArray ()>
array(30701.46894588)
Attributes:
    acceptance_fraction:  0.3945333333333333
<xarray.DataArray ()>
array(30701.92375578)
Attributes:
    acceptance_fraction:  0.3945333333333333


In [102]:
# Use .burn_in() to chop off data before a specific sample number
cut_number = 100
burnt_results6 = results6.burn_in(cut_number) 
#120 seems good for 1000 somewhat random start
#300 for 2000 chain random start
#400 seems good for 3000 chain random start
plt.figure()
plt.title(f'Post burn-in logprob (cut off first {cut_number})')
plt.ylabel('lnprob')
plt.xlabel('sample (chain)')
#ids = [1,3,8,9,10] (for 3000 chain)
for i in range(4):
    plt.plot(burnt_results6.lnprobs[i])
plt.savefig(SAVEPATH + '/few_lnprobs.png')
# to do more systematically could maybe calculate some sort of slope vs maximum slope cutoff?

In [38]:
# looking at the fits for all the different walkers it's clear that several don't converge well
plt.figure()
plt.title('Post burn-in logprob showing bad fits')
plt.ylabel('lnprob')
plt.xlabel('sample (chain)')
for i in range(4):
    plt.plot(burnt_results5.lnprobs[i])
plt.savefig(SAVEPATH + '/few_lnprobs_to_show_bad_fits.png')

In [33]:
new_sample_length = len(burnt_results5.lnprobs[i])
print(burnt_results5.lnprobs[i][new_sample_length-1])

<xarray.DataArray ()>
array(30695.78594419)
Attributes:
    acceptance_fraction:  0.18866666666666668


### Visualize Data Traces

In [104]:
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
#plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/many_theta_fits.png')

In [418]:
# plot theta mod pi
plt.figure()
plt.title('Theta fit mod pi')
plt.ylabel('Theta mod pi')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
plt.axhline(y=np.pi, color='gray', linestyle='-.', label='pi')
for i in range(30):
    theta = samples[i].sel(parameter='theta')
    for j in range(len(theta)):
        if theta[j] < -2:
            theta[j] = theta[j] + 2*np.pi
        elif theta[j] < 0.1:
            theta[j] = theta[j] + np.pi
    plt.plot(theta)
plt.savefig(SAVEPATH + '/many_theta_mod_pi_fits.png')

In [106]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
#plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/few_phi_fits.png')

In [38]:
# 18 starts at 2*pi for phi and 3 has phi off by pi
plt.figure()
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
plt.plot(samples[3].sel(parameter='gap'))

In [45]:
plt.figure()
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
plt.plot(samples[3].sel(parameter='phi'))

In [42]:
plt.figure()
plt.plot(burnt_results5.lnprobs[3])

In [695]:
# plot phi mod 2*pi
plt.figure()
plt.title('Phi fit mod 2pi')
plt.ylabel('Phi mod 2pi')
plt.xlabel('sample (chain)')
two_pi = 2*np.pi
plt.axhline(y=PHI+two_pi, color='gray', linestyle='--', label='real value')
swapped_index = []
for i in range(30):
    phi = results5.samples[i].sel(parameter='phi')
    for j in range(len(phi)):
        if phi[j] < 0.1:
            phi[j] = phi[j] + two_pi
    if all(phi > 4):
        plt.plot(phi)
    else:
        swapped_index.append(i)
plt.savefig(SAVEPATH + '/many_phi_mod_2*pi_fits.png')
print(swapped_index)

[]


In [108]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/many_gap_fits.png')

In [110]:
# look at some traces of r1
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/few_r1_fits.png')

In [113]:
# look at some traces of r2
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/many_r2_fits.png')

In [115]:
# look at some traces of n1
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
#plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/few_n1_fits.png')

In [117]:
# look at some traces of n2
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
#plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/many_n2_fits.png')

In [124]:
# look at some traces of x_g
plt.figure()
plt.title("Central x position (um) fit")
plt.ylabel('x (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=Xg_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(results5.samples[i].sel(parameter='x_g'))
plt.savefig(SAVEPATH + '/few_xg_fits.png')

In [126]:
# look at some traces of y_g
plt.figure()
plt.title("Central y position (um) fit")
plt.ylabel('y (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=Yg_TRUE, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(results5.samples[i].sel(parameter='y_g'))
plt.savefig(SAVEPATH + '/many_yg_fits.png')

In [128]:
# look at some traces of z_g
plt.figure()
plt.title("Central z position (um) fit")
plt.ylabel('z (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=Zg_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(results5.samples[i].sel(parameter='z_g'))
plt.savefig(SAVEPATH + '/few_zg_fits.png')

In [96]:
print(samples.sel(parameter='n_2'))

<xarray.DataArray 'samples' (walker: 1000, chain: 50)>
array([[1.6028308 , 1.60276632, 1.60281288, ..., 1.60268985, 1.60284233,
        1.60277122],
       [1.6028308 , 1.60276632, 1.60281281, ..., 1.60268985, 1.60284233,
        1.60277122],
       [1.6028308 , 1.60276632, 1.60281281, ..., 1.60268985, 1.60284233,
        1.60277023],
       ...,
       [1.60679439, 1.60780986, 1.60676059, ..., 1.60808076, 1.60762128,
        1.60725447],
       [1.60678139, 1.60784512, 1.60676059, ..., 1.60808076, 1.60762128,
        1.60725447],
       [1.60677647, 1.60784512, 1.60676059, ..., 1.60808076, 1.60762128,
        1.60725447]])
Coordinates:
    parameter  <U3 'n_2'
Dimensions without coordinates: walker, chain
Attributes:
    acceptance_fraction:  0.40924


In [624]:
# look at some of the traces for walkers that don't converge
plt.plot(samples[7].sel(parameter='theta'))
plt.plot(samples[10].sel(parameter='theta'))

In [46]:
# look at theta that did converge and add pi to them
for i in range(4):
    plt.plot(samples[i].sel(parameter='theta')+np.pi)

In [54]:
plt.plot(samples[7].sel(parameter='phi'))
plt.plot(samples[10].sel(parameter='phi'))

In [57]:
plt.plot(samples[7].sel(parameter='gap'))
plt.plot(samples[10].sel(parameter='gap'))

### Drop bad convergence (ie low lnprob) walkers. Later attempt to fix these fits instead.

In [148]:
# look at distribution of final lnprob values across walkers
new_sample_length = len(burnt_results5.lnprobs[0])
plt.figure()
plt.plot(burnt_results5.lnprobs[:,(new_sample_length-1)])
plt.savefig(SAVEPATH + '/final_lnprob_values_across_walkers')
maxlnprob = max(burnt_results5.lnprobs[:,(new_sample_length-1)])
converged_value = maxlnprob - 0.02*maxlnprob
print(converged_value)

<xarray.DataArray ()>
array(13627.86458101)


In [130]:
# The bad fits are the swapped angles fits (visible in the reorganized pandas dataset)
# -> is there a good way to correct these or should I just drop them?
# Start by implementing a cutoff in lnprobs to drop them and then can work on fixing later
samples = burnt_results5.samples
converged_samples_nan = xr.DataArray()
# 0 doesn't work as a cutoff universally, example, 3000 chain fit 1 needs 15000 as cutoff
bad_fit_index = []
good_fit_index = []
for i in range(len(samples)):
    new_sample_length = len(burnt_results5.lnprobs[i])
    if burnt_results5.lnprobs[i][new_sample_length-1] > converged_value:
        converged_samples_nan = xr.concat([converged_samples_nan,samples[i]],'walker')
        # .append() isn't quite what we want, try to use xarray methods
        good_fit_index.append(i)
    else:
        bad_fit_index.append(i)
converged_samples = converged_samples_nan[1:]
print(converged_samples)
print(len(converged_samples))
print(bad_fit_index)

<xarray.DataArray (walker: 50, chain: 900, parameter: 11)>
array([[[ 1.58375259,  0.66647132, 58.48837879, ...,  1.60017362,
          0.64913276,  0.64476081],
        [ 1.58375259,  0.66647132, 58.48837879, ...,  1.60017362,
          0.64913276,  0.64476081],
        [ 1.58376101,  0.66647103, 58.48898343, ...,  1.60018139,
          0.64913171,  0.64537217],
        ...,
        [ 1.58338314,  0.6618104 , 58.49373195, ...,  1.59971242,
          0.65374996,  0.64344676],
        [ 1.58338314,  0.6618104 , 58.49373195, ...,  1.59971242,
          0.65374996,  0.64344676],
        [ 1.58338314,  0.6618104 , 58.49373195, ...,  1.59971242,
          0.65374996,  0.64344676]],

       [[ 1.58381038,  0.66639128, 58.48645969, ...,  1.60010244,
          0.64907404,  0.64426231],
        [ 1.58381038,  0.66639128, 58.48645969, ...,  1.60010244,
          0.64907404,  0.64426231],
        [ 1.58381038,  0.66639128, 58.48645969, ...,  1.60010244,
          0.64907404,  0.64426231],
...
    

In [ ]:
# could also do this with ds.drop_sel(space=["IN", "IL"]) where we use walker = [drop indexes]

In [49]:
# plot converged samples
# need to modify this for each fit you want to use it for
# converged_id = [0,1,2,3,4,5,6,8,9]
plt.figure()
for id in good_fit_index:
    plt.plot(burnt_results5.lnprobs[id])

NameError: name 'converged_id' is not defined

#### Visualize good vs. bad fit parameter traces

In [437]:
# plot bad fits to see parrellels between them
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in bad_fit_index:
    plt.plot(results5.samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/30089_bad_theta_fits.png')

In [438]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in bad_fit_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/30089_bad_phi_fits.png')

In [439]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in bad_fit_index:
    plt.plot(results5.samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/30089_bad_gap_fits.png')

In [622]:
# plot good fits to see parrellels between them
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in good_fit_index:
    plt.plot(results5.samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/30089_good_theta_fits.png')

In [ ]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in good_fit_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/30089_good_phi_fits.png')

In [ ]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in good_fit_index:
    plt.plot(results5.samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/30089_good_gap_fits.png')

#### Visualize fits with low starting phi values (near boundary)

In [717]:
# look at how many starting phi look like the starting phi that lead to these bad fits
low_start_index = []
for i in range(50):
    if (samples[i,0].sel(parameter='phi') > np.pi) and (samples[i,0].sel(parameter='phi') < 2*np.pi):
        low_start_index.append(i)
print(low_start_index)

[0, 2, 5, 6, 9, 13, 18, 19, 20, 21, 22, 26, 27, 28, 38, 42, 44, 49]


In [613]:
# for this run 16 is bad since it swaps phi to around pi
low_start_index.remove(16)
print(low_start_index)

[0, 1, 2, 3, 5, 6, 7, 8, 9, 18, 22, 23, 25, 26, 27, 31, 36, 38, 40, 42, 43, 44, 45, 47]


In [718]:
plt.figure()
for id in low_start_index:
    plt.plot(burnt_results5.lnprobs[id])
plt.savefig(SAVEPATH + '/low_starting_phi_lnprobs.png')

In [615]:
plt.figure()
for id in low_start_index:
    plt.plot(burnt_results5.lnprobs[id][100:])
plt.savefig(SAVEPATH + '/low_starting_phi_lnprobs_drop_first_220.png')

In [719]:
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/low_starting_phi_theta_fit.png')

In [720]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI+2*np.pi, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/low_starting_phi_phi_fit.png')

In [64]:
# look at some traces of phi that are never off by pi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI+2*np.pi, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    if all(samples[i,:].sel(parameter='phi') > 4):
        plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/low_starting_phi_phi_fit_not_off_by_pi.png')

In [121]:
# look at the end of some traces of phi that are not off by pi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI+2*np.pi, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    if all(samples[i,:].sel(parameter='phi')[-50:-1] > 4):
        plt.plot(samples[i].sel(parameter='phi')[-50:-1])

In [65]:
# plot starting phi positions for low initial conditions that aren't off by mod pi
plt.figure()
plot_varible = []
for id in low_start_index:
    if samples[id,0].sel(parameter='phi') > 4:
        plot_varible.append(samples[id,0].sel(parameter='phi'))
plt.plot(plot_varible)
plt.savefig(SAVEPATH + '/low_starting_phi_plot_of_starting_phi_not_off_by_pi.png')

In [721]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/low_starting_phi_gap_fit.png')

In [722]:
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/low_starting_phi_r1_fit.png')

In [723]:
# look at some traces of r2
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/low_starting_phi_r2_fit.png')

In [724]:
# look at some traces of n1
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/low_starting_phi_n1_fit.png')

In [725]:
# look at some traces of n2
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/low_starting_phi_n2_fit.png')

In [ ]:
# look at the fits that start with lnprob like the bad fits but then jump to good fits
# these are subset of low_start index and so already convered

#### Visualize fits with high starting phi values (near boundary)

In [726]:
# look at how many starting phi look like the starting phi that lead to bad fits
high_start_index = []
for i in range(50):
    if (samples[i,0].sel(parameter='phi') < 1):
        high_start_index.append(i)
print(high_start_index)

[30, 34, 35, 37, 41, 43, 45, 46, 48]


In [727]:
plt.figure()
for id in high_start_index:
    plt.plot(burnt_results5.lnprobs[id])
plt.savefig(SAVEPATH + '/high_starting_phi_lnprobs.png')

In [728]:
plt.figure()
for id in high_start_index:
    plt.plot(burnt_results5.lnprobs[id][100:])
plt.savefig(SAVEPATH + '/high_starting_phi_lnprobs_drop_first_350.png')

In [729]:
# look at some traces of theta with high starting phi
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/high_starting_phi_theta_fit.png')

In [732]:
# look at some traces of phi with high starting phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/high_starting_phi_phi_fit.png')

In [731]:
# look at some traces of gap with high starting phi
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/high_starting_phi_gap_fit.png')

In [733]:
# look at some traces of r_1 with high starting phi
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/high_starting_phi_r1_fit.png')

In [734]:
# look at some traces of r2 with high starting phi
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/high_starting_phi_r2_fit.png')

In [735]:
# look at some traces of n1 with high starting phi
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/high_starting_phi_n1_fit.png')

In [736]:
# look at some traces of n2 with high starting phi
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/high_starting_phi_n2_fit.png')

#### Visualize fits with low starting theta values

In [442]:
# look at how many starting phi look like the starting phi that lead to these bad fits
low_start_index = []
for i in range(30):
    if (results5.samples[i,0].sel(parameter='theta') > np.pi) and (results5.samples[i,0].sel(parameter='phi') < 2*np.pi):
        low_start_index.append(i)
print(low_start_index)

[2, 5, 9, 12, 14, 15, 16, 19, 20, 21, 23, 24]


In [443]:
plt.figure()
for id in low_start_index:
    plt.plot(burnt_results5.lnprobs[id])
plt.savefig(SAVEPATH + '/boundary_starting_theta_lnprobs.png')

In [446]:
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(results5.samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_theta_fit.png')

In [447]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_phi_fit.png')

In [449]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_gap_fit.png')

In [450]:
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_r1_fit.png')

In [451]:
# look at some traces of r2
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_r2_fit.png')

In [452]:
# look at some traces of n1
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_n1_fit.png')

In [453]:
# look at some traces of n2
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_n2_fit.png')

### Decimate data so just keep independent fits

In [131]:
# look at autocorrelation to see how long it is before lose memory so can decimate into independent samples
series = pd.Series(converged_samples[1].sel(parameter='gap'))
autocorr_as_function_of_time = []
for i in range(len(series)):
    autocorr = series.autocorr(lag=i)
    autocorr_as_function_of_time.append(autocorr)
plt.figure()
plt.plot(autocorr_as_function_of_time)

/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2493: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)


In [167]:
# look at what corresponding trace looks like
plt.figure()
plt.plot(converged_samples[1].sel(parameter='gap'))

In [132]:
# look at autocorrelation of gap data from different walkers
plt.figure()
plt.title('autocorrelation of gap')
for n in range(len(converged_samples)):
    series = pd.Series(converged_samples[n].sel(parameter='gap'))
    autocorr_as_function_of_time = []
    for i in range(len(series)):
        autocorr = series.autocorr(lag=i)
        autocorr_as_function_of_time.append(autocorr)
    plt.plot(autocorr_as_function_of_time)
plt.savefig(SAVEPATH + '/auto_correlation_of_gap.png')

In [221]:
# look at trace of gap from different walkers
plt.figure()
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='gap'))

In [130]:
# look at autocorrelation of theta data from different walkers
plt.figure()
for n in range(len(converged_samples)):
    series = pd.Series(converged_samples[n].sel(parameter='theta'))
    autocorr_as_function_of_time = []
    for i in range(len(series)):
        autocorr = series.autocorr(lag=i)
        autocorr_as_function_of_time.append(autocorr)
    plt.plot(autocorr_as_function_of_time)

In [171]:
# look at trace of theta from different walkers
plt.figure
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='theta'))

In [185]:
# look at trace of n_1 from different walkers
plt.figure
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='n_1'))

In [103]:
# We can use the following notation to cycle throught the different parameter labels
for parameter in converged_samples.coords['parameter'].data:
    print(parameter)

n_1
r_1
x_g
gap
phi
theta
y_g
z_g
n_2
r_2
alpha


In [133]:
# Plot autocorrelation for all the different parameters
# set up plotting
number_columns = int(np.ceil(len(converged_samples.coords['parameter'].data)/3))
fig,axes = plt.subplots(3,number_columns)
m = 1
# go through analysis
for parameter_name in converged_samples.coords['parameter'].data:
    for n in range(len(converged_samples)):
        series = pd.Series(converged_samples[n].sel(parameter=parameter_name))
        autocorr_as_function_of_time = []
        for i in range(len(series)):
            autocorr = series.autocorr(lag=i)
            autocorr_as_function_of_time.append(autocorr)
        plt.subplot(3,number_columns,m)
        plt.plot(autocorr_as_function_of_time)
    # label plot
    row = int(np.floor((m-1)/4))
    column = int(m-4*row)
    axes[row,column-1].set_title(parameter_name)
    fig.supxlabel('Chain Number')
    fig.supylabel('Autocorrelation')
    # increment number tracker
    m = m + 1
plt.savefig(SAVEPATH + '/all_autocorrelation.png')

In [134]:
# find the correlation time for each of these parameters (ie when autocorrelation drops to 0 for the first time)
all_parameter_indices = []
for parameter_name in converged_samples.coords['parameter'].data:
    all_walker_indices = []
    for n in range(len(converged_samples)):
        series = pd.Series(converged_samples[n].sel(parameter=parameter_name))
        autocorr_as_function_of_time = []
        for i in range(len(series)):
            autocorr = series.autocorr(lag=i)
            autocorr_as_function_of_time.append(autocorr)
            if autocorr < 0:
                index = i
                all_walker_indices.append(index)
                break
            elif i == (len(series)-1):
                index = i
                all_walker_indices.append(index)
                print("sample " + str(n) + " of " +  parameter_name + " remains correlated")
    all_parameter_indices.append(all_walker_indices)
print(all_parameter_indices)
# Q: should I find max correlation time or mean correlation time for each parameter?
# start with easiest which is just overall max
overall_correlation_time = np.max(all_parameter_indices)
print(overall_correlation_time)

[[174, 110, 95, 260, 78, 195, 145, 244, 236, 80, 97, 161, 126, 227, 258, 111, 101, 288, 115, 142, 206, 116, 157, 123, 195, 193, 160, 118, 63, 165, 231, 257, 173, 107, 269, 237, 104, 166, 153, 127, 393, 182, 146, 175, 196, 227, 46, 130, 145, 96], [545, 384, 287, 425, 305, 445, 444, 474, 886, 377, 798, 698, 398, 452, 766, 362, 484, 622, 417, 584, 352, 336, 528, 527, 353, 405, 447, 342, 348, 685, 729, 385, 315, 617, 603, 393, 545, 347, 333, 758, 358, 318, 434, 517, 846, 532, 874, 823, 425, 876], [491, 441, 153, 238, 285, 415, 179, 598, 233, 299, 434, 449, 162, 254, 172, 223, 226, 536, 215, 242, 263, 272, 663, 696, 364, 511, 407, 245, 477, 142, 230, 666, 251, 389, 646, 249, 147, 260, 306, 200, 185, 157, 226, 232, 466, 486, 858, 386, 266, 749], [121, 168, 112, 163, 209, 162, 110, 123, 189, 107, 56, 82, 337, 108, 134, 155, 185, 158, 69, 144, 109, 207, 102, 140, 120, 135, 178, 260, 133, 116, 109, 133, 185, 217, 143, 172, 74, 117, 327, 206, 121, 200, 90, 102, 94, 126, 128, 106, 235, 91], [220,

In [135]:
# other approach where we find the mean and then take the max
mean_parameter_corr_time = np.mean(all_parameter_indices, axis=1)
max_of_mean_parameter_corr_time = int(np.max(mean_parameter_corr_time))
print(max_of_mean_parameter_corr_time)

510


In [136]:
# now use correlation time to decimate the data
chain_number = len(converged_samples[0])
number_ind_samples_per_walker = int(np.ceil(chain_number/overall_correlation_time))
independent_samples =[]
for i in range(number_ind_samples_per_walker):
    index = (chain_number-1-overall_correlation_time*i)
    if i==0:
        independent_samples = converged_samples[:,index]
    else:
        independent_samples = xr.concat([independent_samples,(converged_samples[:,index])], 'walker')

In [137]:
# now convert independent samples into a form that can be input into seaborn pairplot
independent_samples_pd = independent_samples.to_dataframe(name = 'independent samples')
independent_samples_pd = independent_samples_pd.reset_index()
independent_samples_pd = independent_samples_pd.pivot('walker','parameter','independent samples')
#print(independent_samples_pd)

/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_88403/3190556392.py:4: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  independent_samples_pd = independent_samples_pd.pivot('walker','parameter','independent samples')


In [138]:
full_pair_plot = sns.pairplot(independent_samples_pd)
full_pair_plot.savefig(SAVEPATH + '/pair_plot_of_all_samples')

In [139]:
# seems like the parameter sets from the end of the run and the beginning of the run
# cluster in different ways, lets try seperating them

# get parameters at the end of the fitting process
end_of_run_ind = independent_samples[:len(converged_samples)]
end_of_run_ind_pd = end_of_run_ind.to_dataframe(name = 'independent samples')
end_of_run_ind_pd = end_of_run_ind_pd.reset_index()
end_of_run_ind_pd = end_of_run_ind_pd.pivot('walker','parameter','independent samples')
print(end_of_run_ind_pd)

# get parameters from earlier on in the fitting process (~correlation time before the end)
early_run_ind = independent_samples[len(converged_samples):]
early_run_ind_pd = early_run_ind.to_dataframe(name = 'independent samples')
early_run_ind_pd = early_run_ind_pd.reset_index()
early_run_ind_pd = early_run_ind_pd.pivot('walker','parameter','independent samples')
#print(early_run_ind_pd)

parameter     alpha       gap       n_1       n_2       phi       r_1  \
walker                                                                  
0          0.643447  0.108364  1.583383  1.599712  6.332153  0.661810   
1          0.646470  0.108649  1.583135  1.599502  6.333560  0.661950   
2          0.645081  0.107495  1.583798  1.599525  6.329234  0.660434   
3          0.643942  0.112076  1.583773  1.599065  6.332690  0.662174   
4          0.642295  0.109880  1.583438  1.598949  6.329080  0.661301   
5          0.645042  0.116856  1.583539  1.599579  6.327833  0.661407   
6          0.644757  0.109983  1.583091  1.599134  6.328076  0.661630   
7          0.645884  0.107676  1.583165  1.599449  6.332342  0.661031   
8          0.645398  0.111168  1.583465  1.599016  6.332288  0.660771   
9          0.642834  0.108255  1.583110  1.599213  6.328696  0.661436   
10         0.646065  0.106528  1.583708  1.599210  6.332349  0.661215   
11         0.644814  0.104259  1.583384  1.598990  

/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_88403/2063201981.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  end_of_run_ind_pd = end_of_run_ind_pd.pivot('walker','parameter','independent samples')
/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_88403/2063201981.py:15: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  early_run_ind_pd = early_run_ind_pd.pivot('walker','parameter','independent samples')


In [140]:
end_of_run_pair_plot = sns.pairplot(end_of_run_ind_pd)
end_of_run_pair_plot.savefig(SAVEPATH + '/13627_pair_plot_of_end_of_run_samples')
# hmm seems like one of the fits is comparatively bad and is an outlier (for 1000 chain, random starting)

In [141]:
early_run_pair_plot = sns.pairplot(early_run_ind_pd)
early_run_pair_plot.savefig(SAVEPATH + '/13627_cutoff_pair_plot_of_early_run_samples')

In [142]:
# return real fit values
starting_means = []
for p in model6._parameters:
        starting_means.append(p.mu)
print(starting_means)

[1.5837615975996668, 0.666459947416345, 58.484832600004395, 0.11289545130974342, 6.3318712339240255, 2.6508866488913227, 108.65291035266634, 21.054742080951772, 1.6001721492809295, 0.6491023121734101, 0.6465348913785244]


In [144]:
# compare average of end points of converged fits with ground truth values
mean_prediction = np.mean(end_of_run_ind_pd,axis=0)
print(mean_prediction)
print(starting_means)
mean_differences = np.zeros(len(starting_means))
corresponding_id = [2,5,8,1,4,7,9,10,3,6,0]
for i in range(len(starting_means)):
    mean_differences[i] = starting_means[i]-mean_prediction[corresponding_id[i]]
print(mean_differences)

parameter
alpha      0.644564
gap        0.109737
n_1        1.583458
n_2        1.599162
phi        6.330971
r_1        0.661251
r_2        0.654057
theta      2.654173
x_g       58.493734
y_g      108.651966
z_g       21.030329
dtype: float64
[1.5837615975996668, 0.666459947416345, 58.484832600004395, 0.11289545130974342, 6.3318712339240255, 2.6508866488913227, 108.65291035266634, 21.054742080951772, 1.6001721492809295, 0.6491023121734101, 0.6465348913785244]
[ 0.00030376  0.0052094  -0.00890111  0.00315829  0.00090066 -0.00328601
  0.00094411  0.02441259  0.00101032 -0.00495494  0.00197072]


In [145]:
mean_prediction = np.mean(end_of_run_ind_pd,axis=0)
mean_differences = mean_prediction.copy()
opp_corresponding_id = [10,3,0,8,4,1,9,5,2,6,7]
for i in range(len(starting_means)):
    mean_differences[i] = starting_means[opp_corresponding_id[i]]-mean_prediction[i]
print(mean_prediction)
print(mean_differences)

parameter
alpha      0.644564
gap        0.109737
n_1        1.583458
n_2        1.599162
phi        6.330971
r_1        0.661251
r_2        0.654057
theta      2.654173
x_g       58.493734
y_g      108.651966
z_g       21.030329
dtype: float64
parameter
alpha    0.001971
gap      0.003158
n_1      0.000304
n_2      0.001010
phi      0.000901
r_1      0.005209
r_2     -0.004955
theta   -0.003286
x_g     -0.008901
y_g      0.000944
z_g      0.024413
dtype: float64


In [86]:
type(mean_differences)

pandas.core.series.Series

In [146]:
mean_differences.to_csv(SAVEPATH+'/13627_cutoff_initial_values_minus_mean_of_fits')

In [471]:
# test is cos(phi-previous_phi) could be leading to problems in lnprior
phi = [0,2*np.pi,2*np.pi,4*np.pi,4*np.pi,0]
previous_phi = [np.pi/4,np.pi/4,9*np.pi/4,np.pi/4,9*np.pi/4,-7*np.pi/4]
for i in range(len(phi)):
    print(np.cos(phi[i]-previous_phi[i]))
# looks like it works totally fine

0.7071067811865476
0.7071067811865474
0.7071067811865476
0.7071067811865466
0.7071067811865474
0.7071067811865474
